In [0]:
%run "../config/00_config"

In [0]:
%run "../config/tables/ecommerce_rastreamento_entregas_config"

In [0]:
%run "../utils/00_utils"

In [0]:
adls_options = build_adls_options(
    storage_account_name=ADLS_STORAGE_ACCOUNT_NAME,
    client_id=ADLS_CLIENT_ID,
    tenant_id=ADLS_TENANT_ID,
    client_secret=ADLS_CLIENT_SECRET,
)

print("Configuração da análise:")
print(f"Arquivo origem: {SOURCE_FILE}")
print(f"Caminho origem: {SOURCE_PATH}")
print(f"Tabela destino configurada: {TARGET_FULL_TABLE}")

In [0]:
df_source = read_source_csv(
    spark=spark,
    source_path=SOURCE_PATH,
    adls_options=adls_options,
    csv_options=CSV_OPTIONS,
)

print("Arquivo fonte lido com sucesso.")

In [0]:
source_overview = get_dataframe_overview(df_source)

print("Resumo da origem:")
print(f"Total de linhas: {source_overview['total_linhas']}")
print(f"Total de colunas: {source_overview['total_colunas']}")

print("Colunas encontradas:")
for column_name in source_overview["colunas"]:
    print(f"- {column_name}")

In [0]:
df_source.printSchema()

In [0]:
display(df_source.limit(10))

In [0]:
display(get_null_summary(df_source))

In [0]:
from pyspark.sql.functions import countDistinct, col

total_rows = df_source.count()

column_profile = []

for column_name in df_source.columns:
    null_count = df_source.filter(col(column_name).isNull()).count()
    distinct_count = df_source.select(countDistinct(col(column_name))).collect()[0][0]

    column_profile.append({
        "coluna": column_name,
        "tipo": dict(df_source.dtypes)[column_name],
        "total_linhas": total_rows,
        "qtd_nulos": null_count,
        "qtd_distintos": distinct_count,
        "possivel_chave": null_count == 0 and distinct_count == total_rows,
    })

df_column_profile = spark.createDataFrame(column_profile)

display(df_column_profile)

In [0]:
print("Sugestão para preencher EXPECTED_COLUMNS no 00_config:")
print("EXPECTED_COLUMNS = [")

for column_name in df_source.columns:
    print(f'    "{column_name}",')

print("]")

In [0]:
candidate_key_columns = [
    row["coluna"]
    for row in df_column_profile.collect()
    if row["possivel_chave"]
]

print("Possíveis colunas candidatas a chave:")
for column_name in candidate_key_columns:
    print(f"- {column_name}")

if candidate_key_columns:
    print("")
    print("Sugestão inicial para o 00_config:")
    print(f'KEY_COLUMNS = ["{candidate_key_columns[0]}"]')
else:
    print("Nenhuma coluna candidata a chave única foi identificada automaticamente.")

In [0]:
print("Configuração atual de validações:")
print(f"EXPECTED_COLUMNS preenchido? {bool(EXPECTED_COLUMNS)}")
print(f"KEY_COLUMNS preenchido? {bool(KEY_COLUMNS)}")
print(f"VALIDATE_EXPECTED_COLUMNS: {VALIDATE_EXPECTED_COLUMNS}")
print(f"VALIDATE_KEY_COLUMNS: {VALIDATE_KEY_COLUMNS}")

if EXPECTED_COLUMNS:
    print("")
    print("EXPECTED_COLUMNS atual:")
    print(EXPECTED_COLUMNS)

if KEY_COLUMNS:
    print("")
    print("KEY_COLUMNS atual:")
    print(KEY_COLUMNS)

In [0]:
columns_validation = validate_required_columns(
    df=df_source,
    expected_columns=EXPECTED_COLUMNS,
)

print(columns_validation["message"])

if columns_validation["unexpected_columns"]:
    print(f"Colunas adicionais encontradas: {columns_validation['unexpected_columns']}")

In [0]:
key_validation = validate_key_columns(
    df=df_source,
    key_columns=KEY_COLUMNS,
)

print(key_validation["message"])

if key_validation["validation_applied"]:
    print(f"Colunas de chave validadas: {key_validation['key_columns']}")
    print(f"Nulos por coluna de chave: {key_validation['null_counts']}")
    print(f"Registros duplicados pela chave: {key_validation['duplicated_rows']}")

In [0]:
try:
    df_target = read_sql_table(
        spark=spark,
        sql_host=SQL_HOST,
        sql_database=SQL_DATABASE,
        sql_username=SQL_USERNAME,
        sql_password=SQL_PASSWORD,
        table_name=TARGET_FULL_TABLE,
        sql_port=SQL_PORT,
    )

    print(f"Tabela destino lida com sucesso: {TARGET_FULL_TABLE}")

except Exception as error:
    df_target = None
    print(f"Não foi possível ler a tabela destino: {TARGET_FULL_TABLE}")
    print("Isso pode acontecer se o notebook de ingestion ainda não foi executado.")
    print(f"Erro retornado: {error}")

In [0]:
if df_target is not None:
    target_overview = get_dataframe_overview(df_target)

    print("Resumo do destino SQL:")
    print(f"Total de linhas: {target_overview['total_linhas']}")
    print(f"Total de colunas: {target_overview['total_colunas']}")

    print("Colunas encontradas no destino:")
    for column_name in target_overview["colunas"]:
        print(f"- {column_name}")

    df_target.printSchema()
else:
    print("Tabela destino não disponível para análise.")

In [0]:
if df_target is not None:
    display(df_target.limit(10))
else:
    print("Tabela destino não disponível para exibição.")

In [0]:
if df_target is not None:
    count_validation = compare_row_counts(
        source_df=df_source,
        target_df=df_target,
    )

    print(count_validation["message"])
    print(f"Total de registros na origem: {count_validation['source_count']}")
    print(f"Total de registros no destino: {count_validation['target_count']}")
else:
    print("Comparação origem vs destino ignorada porque a tabela destino não foi lida.")